# PyTorch Neural Network para Regressão

Construção de uma MLP com `torch.nn`, ciclo de treino com **validação** e avaliação de desempenho.

### Componentes principais
| Componente | Escolha |
|---|---|
| Camada de saída | `nn.Linear(..., 1)` — sem ativação (valor contínuo) |
| Loss | `nn.MSELoss` |
| Otimizador | `Adam` |
| Métrica | MSE / RMSE / R² |

## 1. Importar Bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Reprodutibilidade (Cap. 3 do livro)
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch versão: {torch.__version__}")

## 2. Gerar e Preparar Dados

Os dados são normalizados com `StandardScaler` — prática essencial para normalizar as entradas e acelerar a convergência.

Os dados são divididos em **treino** (usado para atualizar pesos) e **teste** (avaliação final).

In [ ]:
# Dados sintéticos: y = 3x0 + 2x1 - x2 + 0.5x3 + ruído
X = np.random.randn(300, 5)
y = 3*X[:,0] + 2*X[:,1] - X[:,2] + 0.5*X[:,3] + np.random.randn(300)*0.1

# 80% treino, 20% teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalização — prática essencial para acelerar a convergência
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Converter para tensores PyTorch
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
X_test  = torch.FloatTensor(X_test)
y_test  = torch.FloatTensor(y_test).reshape(-1, 1)

print(f"Treino : X={X_train.shape}, y={y_train.shape}")
print(f"Teste  : X={X_test.shape},  y={y_test.shape}")

## 3. Arquitetura da Rede Neural (MLP)

Uma MLP é uma composição de transformações afins seguidas de ativações não-lineares:

```
x₁ = ReLU(W₀·x  + b₀)
x₂ = ReLU(W₁·x₁ + b₁)
ŷ  =       W₂·x₂ + b₂   ← sem ativação (saída contínua)
```

**Por que `nn.Sequential`?** — forma compacta e legível de encadear camadas.

**Por que ReLU?** — atualmente considerada a melhor função de ativação geral; evita problemas de gradientes saturados com tanh/sigmoid.

In [ ]:
class RegressionNN(nn.Module):
    """
    MLP para Regressão
    Arquitetura: 5 → 64 → 32 → 1  (saída linear = valor contínuo qualquer)
    """
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        # nn.Sequential: forma compacta de compor camadas
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),                          # ativação não-linear
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)                    # sem ativação → saída real irrestrita
        )

    def forward(self, x):
        return self.network(x)

model = RegressionNN(input_size=5, hidden_size=64)
print(model)

# Inspecionar parâmetros
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal de parâmetros treináveis: {n_params:,}")

## 4. Loss e Otimizador

- **`nn.MSELoss`**: Erro Quadrático Médio — loss padrão para regressão.
- **`Adam`**: otimizador adaptativo, muito menos sensível à escala dos parâmetros do que SGD.  
  Permite usar a taxa de aprendizado original sem normalização extra.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function : {criterion}")
print(f"Otimizador    : Adam  (lr=0.001)")

## 5. Ciclo de Treino com Acompanhamento de Validação

Ciclo padrão:

```
model.train()           # habilita batchnorm/dropout para treino
─── forward pass ───    # calcula predições
─── loss ───────────    # compara com gabarito
optimizer.zero_grad()   # zera gradientes acumulados
loss.backward()         # autograd computa gradientes
optimizer.step()        # Adam atualiza pesos

model.eval()
with torch.no_grad():   # desliga autograd na validação
    val_loss = ...
```

> **Regra:** se `train_loss` cair mas `val_loss` subir → **overfitting**.

In [ ]:
epochs      = 200
train_losses = []
val_losses   = []

for epoch in range(epochs):

    # ── Fase de Treino ────────────────────────────────────────────────
    model.train()                         # habilita modo treino
    outputs = model(X_train)
    loss    = criterion(outputs, y_train)

    optimizer.zero_grad()                 # zera gradientes
    loss.backward()                       # retropropagação automática (autograd)
    optimizer.step()                      # Adam atualiza os pesos
    train_losses.append(loss.item())

    # ── Fase de Validação — sem gradientes ────────────────────────────
    model.eval()
    with torch.no_grad():                 # desliga autograd → economiza memória
        val_out  = model(X_test)
        val_loss = criterion(val_out, y_test)
    val_losses.append(val_loss.item())

    if (epoch + 1) % 40 == 0:
        print(f"Epoch [{epoch+1:3d}/{epochs}] | "
              f"Train Loss: {loss.item():.4f} | "
              f"Val Loss: {val_loss.item():.4f}")

print("\nTreinamento concluído!")

## 6. Avaliação e Visualização

Calculamos MSE, RMSE e R² para treino e teste, e verificamos graficamente se o modelo
generaliza bem — sem overfitting.

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_train = model(X_train).numpy()
    y_pred_test  = model(X_test).numpy()

# Métricas
mse_train  = mean_squared_error(y_train.numpy(), y_pred_train)
rmse_train = np.sqrt(mse_train)
r2_train   = r2_score(y_train.numpy(), y_pred_train)

mse_test   = mean_squared_error(y_test.numpy(), y_pred_test)
rmse_test  = np.sqrt(mse_test)
r2_test    = r2_score(y_test.numpy(), y_pred_test)

print("=== Métricas de Desempenho ===")
print(f"{'Conjunto':<10} {'MSE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 38)
print(f"{'Treino':<10} {mse_train:>8.4f} {rmse_train:>8.4f} {r2_train:>8.4f}")
print(f"{'Teste':<10} {mse_test:>8.4f}  {rmse_test:>8.4f} {r2_test:>8.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: Treino vs Validação ───────────────────────────────────────────────
axes[0].plot(train_losses, label='Treino',    color='royalblue')
axes[0].plot(val_losses,   label='Validação', color='darkorange', linestyle='--')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Curvas de Loss\n(Verificação de Overfitting)')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# ── Plot 2: Predições vs Real — Treino ───────────────────────────────────────
axes[1].scatter(y_train.numpy(), y_pred_train, alpha=0.5, color='royalblue', s=20)
lim = [float(y_train.min()), float(y_train.max())]
axes[1].plot(lim, lim, 'r--', lw=2, label='Perfeito')
axes[1].set_xlabel('Valores Reais')
axes[1].set_ylabel('Predições')
axes[1].set_title(f'Treino — Predições vs Real\nR² = {r2_train:.4f}')
axes[1].legend()
axes[1].grid(True, alpha=0.4)

# ── Plot 3: Predições vs Real — Teste ────────────────────────────────────────
axes[2].scatter(y_test.numpy(), y_pred_test, alpha=0.6, color='darkorange', s=20)
lim = [float(y_test.min()), float(y_test.max())]
axes[2].plot(lim, lim, 'r--', lw=2, label='Perfeito')
axes[2].set_xlabel('Valores Reais')
axes[2].set_ylabel('Predições')
axes[2].set_title(f'Teste — Predições vs Real\nR² = {r2_test:.4f}')
axes[2].legend()
axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('regressao_resultados.png', dpi=120, bbox_inches='tight')
plt.show()